# VEP DNA Pipeline

In [1]:
# Automatically reload code in notebook
%load_ext autoreload
%autoreload 2

import os
os.chdir(os.path.dirname(os.path.abspath('.')))
import pandas as pd
import polars as pl
import seqpro as sp
import genvarloader as gvl

import sys
from pathlib import Path
from tempfile import TemporaryDirectory

import genvarloader as gvl
import numba as nb
import numpy as np
import polars as pl
import seqpro as sp
import pooch
from loguru import logger
from einops import rearrange
from tqdm.auto import tqdm

/grid/koo/home/schilder/.conda/envs/genome-loader/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Create GVL database

## Download population data

In [2]:
# GRCh38 chromosome 22 sequence
reference = pooch.retrieve(
    url="https://ftp.ensembl.org/pub/release-112/fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.chromosome.22.fa.gz",
    known_hash="sha256:974f97ac8ef7ffae971b63b47608feda327403be40c27e391ee4a1a78b800df5",
    progressbar=True,
)
if not Path(f"{reference[:-3]}.bgz").exists():
    !gzip -dc {reference} | bgzip > {reference[:-3]}.bgz
reference = reference[:-3] + ".bgz"

# PLINK 2 files
variants = pooch.retrieve(
    url="doi:10.5281/zenodo.13656224/1kGP.chr22.pgen",
    known_hash="md5:31aba970e35f816701b2b99118dfc2aa",
    progressbar=True,
    fname="1kGP.chr22.pgen",
)
pooch.retrieve(
    url="doi:10.5281/zenodo.13656224/1kGP.chr22.psam",
    known_hash="md5:eefa7aad5acffe62bf41df0a4600129c",
    progressbar=True,
    fname="1kGP.chr22.psam",
)
pooch.retrieve(
    url="doi:10.5281/zenodo.13656224/1kGP.chr22.pvar",
    known_hash="md5:5f922af91c1a2f6822e2f1bb4469d12b",
    progressbar=True,
    fname="1kGP.chr22.pvar",
)


'/grid/koo/home/schilder/.cache/pooch/1kGP.chr22.pvar'

## Create BED file

This BED file is based on the coordinates of the UTR variants we're planning on injecting, surrounded by 500kb windows.

In [3]:
bed = gvl.read_bedlike("notebooks/snps_500kb_windows.bed")[:20]
bed.head()

chrom,chromStart,chromEnd,name,score,strand
str,i64,i64,str,f64,str
"""22""",16835042,17335042,"""340557""",0.0,"""+"""
"""22""",16835050,17335050,"""340558""",0.0,"""+"""
"""22""",16835053,17335053,"""340559""",0.0,"""+"""
"""22""",16835059,17335059,"""894942""",0.0,"""+"""
"""22""",16835065,17335065,"""340560""",0.0,"""+"""


## Create GVL database

In [4]:
tmp_dir = TemporaryDirectory(suffix=".gvl")
ds_path = tmp_dir.name
gvl.write(
    path=ds_path,
    bed=gvl.with_length(bed, 2**17),  # change region length to 131,072 bp
    variants=variants,
    overwrite=True,
)

2025-05-16 14:57:50.288 | INFO     | genvarloader._dataset._write:write:76 - Writing dataset to /tmp/tmpiul_w1ly.gvl
2025-05-16 14:57:50.289 | INFO     | genvarloader._dataset._write:write:83 - Found existing GVL store, overwriting.
2025-05-16 14:57:50.308 | INFO     | genoray._pgen:_read_index:1054 - Loading genoray index.
2025-05-16 14:57:51.414 | INFO     | genvarloader._dataset._write:write:151 - Using 451 samples.
2025-05-16 14:57:51.416 | INFO     | genvarloader._dataset._write:write:157 - Writing genotypes.
Processing genotypes for 20 regions on contig 22: 100%|██████████| 20/20 [00:01<00:00, 17.02 region/s]
2025-05-16 14:57:52.826 | INFO     | genvarloader._dataset._write:write:181 - Finished writing.
/grid/koo/home/schilder/.conda/envs/genome-loader/lib/python3.12/site-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datet

## Import GVL database

In [5]:
ds = (
    gvl.Dataset.open(ds_path, reference=reference)
    .with_seqs("haplotypes")
    .with_len(2**17)
)

2025-05-16 14:58:12.587 | INFO     | genvarloader._dataset._impl:open:222 - Loading reference genome into memory. This typically has a modest memory footprint (a few GB) and greatly improves performance.
2025-05-16 14:58:12.839 | INFO     | genvarloader._dataset._reconstruct:from_path:298 - Loading variant data.
2025-05-16 14:58:12.880 | INFO     | genvarloader._dataset._impl:open:309 - Opened dataset:
GVL store at /tmp/tmpiul_w1ly.gvl
Is subset: False
# of regions: 20
# of samples: 451
Output length: ragged
Jitter: 0 (max: 0)
Deterministic: True
Sequence type: reference [haplotypes] annotated variants
Active tracks: None
Tracks available: None



## Run VEP

In [ ]:
import os
import datetime
from tqdm.auto import tqdm

models = ["borzoi","Evo2-7B", "SpliceAI"]
sites = gvl.sites_vcf_to_table('./notebooks/filtered_chr22_snps.vcf')
results_dir = 'results'
variant_set = "ClinVar_UTR"
force = False

def some_model(seq_wt, seq_mut):
    return np.random.randn(1)[0]

def get_region_name(region):
    return f'chr{region["chrom"]}:{region["chromStart"]}-{region["chromEnd"]}'

def get_results_path(region_name, sample, model, variant_set):
    return f'{results_dir}/{region_name}/{sample}/{model}/{variant_set}.parquet'

# Set environment variable to suppress datetime warnings
os.environ['PYTHONWARNINGS'] = 'ignore::DeprecationWarning:jupyter_client.session'

# Iterate over each region, which are centered on the SNPs
for region_idx, region in tqdm(enumerate(ds.regions.iter_rows(named=True)),
                               total=len(ds.regions),
                               desc="Iterating over regions"):

    region_name = get_region_name(region)

    # Iterate over each sample
    for sample_idx, sample in tqdm(enumerate(ds.samples),
                                   total=len(ds.samples),
                                    desc="Iterating over samples"):
        seq_wt1 = ds[region_idx, sample_idx][0]
        seq_wt2 = ds[region_idx, sample_idx][1]

        # Iterate over each model
        for model in tqdm(models,
                          desc="Iterating over models"):
            print(model)
            
            # Initialize results list
            results = []

            # Get the results path
            results_path = get_results_path(region_name, sample, model, variant_set)
            
            # Skip if the results already exist and force is False
            if os.path.exists(results_path) and not force:
                continue

            # Subset the sites to the region
            sites_region = sites.filter(
                (pl.col("CHROM") == region["chrom"]) & 
                (pl.col("POS") >= region["chromStart"]) & 
                (pl.col("POS") <= region["chromEnd"])
            )

            # Skip if there are no sites in the region
            if sites_region.shape[0] == 0:
                continue

            # Iterate over each site
            for site_idx, site in enumerate(sites_region.iter_rows(named=True)):

                # Generate the mutated sequence
                site_ds = gvl.DatasetWithSites(ds, sites_region[site_idx]) 
                ann_haps, flags = site_ds[region_idx, sample_idx]
                seq_mut1 = ann_haps.haps[0]
                seq_mut2 = ann_haps.haps[1]

                # Run the model to get the VEP score
                vep = some_model(seq_wt1, seq_mut1)

                # Add the VEP score to the site
                site['VEP'] = vep

                # Add the site-specific results to the list
                results.append(site)
                
            # Save the results
            os.makedirs(os.path.dirname(results_path), exist_ok=True)
            pd.DataFrame(results).to_parquet(results_path)

            break
        break
    break
